# Tutorial - Explicit control with DDSP

*This Colab notebook tutorial is part of the IRCAM formation on "Deep audio generative models in MaxMSP and Abelton Live" (2025). The course materials are available on our [github](https://github.com/NilsDem/creative_mL_maxmsp?tab=readme-ov-file).*

Here, you will learn to train your own DDSP model in `Pytorch` and use it for real-time synthesis in MaxMSP, with explicit controls on pitch and loudness, using  the `nn~` external ([github](https://github.com/acids-ircam/nn_tilde.git)).


## Part 1 - Explicit control with DDSP~

The **Differentiable Digital Signal Processing** (**DDSP**)[1] model was introduced in 2019 by J. Engel et al. from Google Magenta research team. It integrates components from classic signal processing methods into a deep learning framework to enable raw audio waveform synthesis. In particular, a signal model is used to directly control the **pitch** and **loudness** of the generated signal. The code implementation and tutorials are available on their [website](https://magenta.tensorflow.org/ddsp).

As illustrated in the figure bellow (taken from the original paper[1]), the model is composed of DSP components and a deterministic auto-encoder to learn the synthesis parameters through a *latent* representation $\mathbf{z}$. The audio signal is split into successive time frames from which we extract :
- the **pitch**, using a pitch estimation algorithm (in the original paper they rely on **CREPE**[2] which consists of a convolutional neural network),
- the **loudness**, directly computed from the raw waveform using a log-scaled, A-weighted power spectrum,
- the ***latent* representation**, using a Gated Recurrent Unit (GRU) encoder network on the first 30 Mel Frequency Cepstral Coefficients (MFCC) of each frame, that is supposed to capture the remaining signal features associated to timbre.

The sound is generated following Spectral Modelling Synthesis (SMS) which consists of adding the contributions from an additive synthesizer (combination of harmonic oscillators) and a filtered white noise. The extracted features are then fed to the decoder (also a GRU network) to generate the harmonic distribution and the filter coefficients. The extracted pitch is directly used as the fundamental frequency of the harmonic synthesizer.

The model is trained to encode and reconstruct the signal by minimizing the multiscale spectral distance

![DDSP archi](https://storage.googleapis.com/ddsp/additive_diagram/ddsp_autoencoder.png)

Thanks to the DSP components, the model is relatively lightweight and fast to train (~2 hours) with a very limited set of parameters compared to large autoregressive or adversarial models such as WaveNet[3] or GANSynth[4]. Moreover, it can be used to generate 48kHZ audio in real-time on a standard laptop CPU. However, the range of sounds that can be generated is restricted especially to monophonic signals.

Today, you will learn to train your own DDSP model in `Pytorch` and use it for real-time synthesis in MaxMSP, with explicit controls on pitch and loudness, using  the `nn~` external ([github](https://github.com/acids-ircam/nn_tilde.git)).

Our code is mostly based on the implementation provided in [https://github.com/acids-ircam/ddsp_pytorch.git](https://github.com/acids-ircam/ddsp_pytorch.git) which contains the implementation of the DDSP model in `Pytorch` that can be exported into a torchscript model to be used in `PureData`for real-time synthesis. In addition, for this tutorial we:

- added the code implementation to export and use a pretrained DDSP model in MaxMSP using `nn~`,
- implemented a streamable version of `torch.crepe` to extract pitch and loudness signals in real-time in MaxMSP.

In this tutorial we will:
- prepare a workspace that works well in Google Colab, optionally backed by Google Drive for persistence
- download one example dataset
- install the deep learning libraries required for training
- preprocess the dataset for DDSP
- instantiate and train the DDSP model
- export the trained model to TorchScript for use in MaxMSP


### Storage setup
This notebook is designed to run on **Google Colab**. The next cells define a single workspace root so that every path in the notebook follows the same configuration.

Eventually connect to gdrive to keep the files after the session is closed.

In [ ]:
from pathlib import Path

try:
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = Path('/content/drive/MyDrive/ddsp')
  BASE_WORKDIR.mkdir(parents=True, exist_ok=True)

except:
  print("Failed mounting to drive")
  PROJECT_ROOT = Path('.')

DATA_ROOT = PROJECT_ROOT / 'data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data root: {DATA_ROOT}')


# Download a dataset
Choose **one** of the datasets below for your experiment. All downloads use `DATA_ROOT`, so changing the storage location later only requires updating the configuration cell above.


### Vocal Dataset
Source: https://zenodo.org/records/1193957


In [ ]:
VOCAL_DIR = DATA_ROOT / 'vocal_set'
VOCAL_DIR.mkdir(parents=True, exist_ok=True)
VOCAL_ZIP = VOCAL_DIR / 'VocalSet.zip'

if not VOCAL_ZIP.exists():
    !wget -O "{VOCAL_ZIP}" "https://zenodo.org/records/1193957/files/VocalSet.zip?download=1"

!unzip -n "{VOCAL_ZIP}" -d "{VOCAL_DIR}"


DATA_DIR = VOCAL_DIR


### Bach Violin Dataset
Source: https://zenodo.org/records/6050245


In [ ]:
VIOLIN_DIR = DATA_ROOT / 'bach_violin'
VIOLIN_DIR.mkdir(parents=True, exist_ok=True)
VIOLIN_ZIP = VIOLIN_DIR / 'bach-violin-dataset-v1.0.zip'

if not VIOLIN_ZIP.exists():
    !wget -O "{VIOLIN_ZIP}" "https://github.com/salu133445/bach-violin-dataset/releases/download/v1.0/bach-violin-dataset-v1.0.zip"

!unzip -n "{VIOLIN_ZIP}" -d "{VIOLIN_DIR}"

DATA_DIR = VIOLIN_DIR


### Guitar Dataset
Source: https://zenodo.org/records/3371780


In [ ]:
GUITAR_DIR = DATA_ROOT / 'guitar'
GUITAR_DIR.mkdir(parents=True, exist_ok=True)
GUITAR_ZIP = GUITAR_DIR / 'audio_mono-mic.zip'

if not GUITAR_ZIP.exists():
    !wget -O "{GUITAR_ZIP}" "https://zenodo.org/records/3371780/files/audio_mono-mic.zip?download=1"

!unzip -n "{GUITAR_ZIP}" -d "{GUITAR_DIR}"

DATA_DIR = GUITAR_DIR


# Setup your environment and install package requirements
Place the `ddsp_pytorch` folder inside `PROJECT_ROOT`.

On Google Colab + Drive, the recommended layout is:
- `MyDrive/ddsp/ddsp_pytorch` for the project code
- `MyDrive/ddsp/data/...` for the downloaded datasets

Once the folder is in place, the next cell defines the project paths used by the rest of the notebook.


In [ ]:
MAIN_DIR = PROJECT_ROOT / 'ddsp_pytorch'

if not MAIN_DIR.exists():
    raise FileNotFoundError(
        f'Could not find {MAIN_DIR}. Copy the ddsp_pytorch folder there before continuing.'
    )

RUNS_ROOT = Path(MAIN_DIR / 'runs')
EXPORTS_DIR = Path(MAIN_DIR / 'exports')

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'MAIN_DIR: {MAIN_DIR}')


In [ ]:
print(f'Current workspace root: {PROJECT_ROOT}')

Now install the Python dependencies required for preprocessing, training, and export. On a fresh Colab runtime this step can take a few minutes.


In [ ]:
import os
os.chdir(MAIN_DIR)
print(f'Working directory: {os.getcwd()}')

!pip install -r requirements.txt

# Some imports

In [ ]:
import torch
import pathlib
import librosa
import numpy as np
import torchcrepe
import torchaudio
import os
import nn_tilde
import soundfile as sf

from IPython.display import display, Audio
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Define the main parameters of our model
This is also where we point the notebook to the dataset we want to preprocess and train on.


In [ ]:
MODEL_NAME = "violin_test"
SR = 44100  # Sampling rate
BLOCK_SIZE = 512  # Size of individual blocks processed by the model
SIGNAL_LENGTH = 131072  # Length of signals for training
CAPACITY = 512  # Size of the model (use a power of 2)

extensions = ["wav", "mp3", "flac", "opus"]
PROCESSED_DIR = MAIN_DIR / f"processed_{MODEL_NAME}"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)
print("Training audio path:", DATA_DIR)

# Preprocess the data

Before training the DDSP model, we need to process the raw audio as follows:
- resample the audio files to the model sampling rate
- cut the audio into chunks with a fixed length
- extract pitch for each chunk with TorchCrepe
- extract loudness from an A-weighted spectrum

The processed arrays will be saved to disk so you can skip this step in later sessions.


### Some useful functions

Find all the audio files in a given directory

In [ ]:
def get_files(data_location, extensions, **kwargs):
    all_files = []
    for ext in extensions:
        all_files.extend(list(pathlib.Path(data_location).rglob(f"*.{ext}")))



    all_files = [ f for f in all_files if "MAC" not in str(f)]

    return all_files


Functions to extract pitch and loudness from an audio sample

In [ ]:
def extract_pitch(signal, sampling_rate, block_size, device="cpu"):

    length = signal.shape[-1] // block_size

    f0 = torchcrepe.predict(
        torch.from_numpy(signal).reshape(1, -1),
        sampling_rate,
        block_size,
        fmin=50,
        fmax=2000,
        model="tiny",
        batch_size=512,
        device=device,
    )

    f0 = f0[0, :-1].cpu().numpy()

    if f0.shape[-1] != length:
        f0 = np.interp(
            np.linspace(0, 1, length, endpoint=False),
            np.linspace(0, 1, f0.shape[-1], endpoint=False),
            f0,
        )

    return f0


In [ ]:
def extract_loudness(signal,
                     sampling_rate,
                     block_size,
                     n_fft=2048,
                     device="cpu"):

    ts = torchaudio.transforms.Spectrogram(n_fft=n_fft,
                                           hop_length=block_size,
                                           win_length=n_fft,
                                           center=True).to(device)

    S = ts(torch.from_numpy(signal).to(device)).cpu()

    S = torch.log(abs(S) + 1e-7)

    f = librosa.fft_frequencies(sr=sampling_rate, n_fft=n_fft)
    a_weight = librosa.A_weighting(f)

    S = S + torch.from_numpy(a_weight).reshape(-1, 1)
    S = torch.mean(S, 0)[..., :-1]

    return S

Full processing function for one audio file. Audio is padded to be a multiple of the signal length.

In [ ]:
def preprocess(f, sampling_rate, block_size, signal_length, device, **kwargs):
    x, sr = librosa.load(path=f, sr=sampling_rate)

    N = (signal_length - len(x) % signal_length) % signal_length
    x = np.pad(x, (0, N))

    loudness = extract_loudness(x, sampling_rate, block_size, device=device)
    pitch = extract_pitch(x, sampling_rate, block_size, device=device)

    x = x.reshape(-1, signal_length)
    pitch = pitch.reshape(x.shape[0], -1)
    loudness = loudness.reshape(x.shape[0], -1)

    return x, pitch, loudness


### Preprocessing loop
This step can take a while, especially on larger datasets. If you are testing the notebook for the first time, start with a small value of `N_FILES` and increase it later.


In [ ]:
files = get_files(DATA_DIR, extensions)

print(len(files), " files found")

To speed up processing, you can only use a limited number of files from the dataset. To use all the files, set N_files to __None__.

In [ ]:
N_FILES = 10

progress_bar = tqdm(files[:N_FILES])

signals = []
pitchs = []
loudness = []

for i, f in enumerate(progress_bar):
    progress_bar.set_description(str(f))
    x, p, l = preprocess(f,
                         sampling_rate=SR,
                         block_size=BLOCK_SIZE,
                         signal_length=SIGNAL_LENGTH,
                         device=device)
    signals.append(x)
    pitchs.append(p)
    loudness.append(l)

#### Save the processed data for future experiments
The arrays are stored in `PROCESSED_DIR`, which makes it easy to resume later without recomputing pitch and loudness.


In [ ]:
signals = np.concatenate(signals, 0).astype(np.float32)
pitchs = np.concatenate(pitchs, 0).astype(np.float32)
loudness = np.concatenate(loudness, 0).astype(np.float32)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

np.save(PROCESSED_DIR / "signals.npy", signals)
np.save(PROCESSED_DIR / "pitchs.npy", pitchs)
np.save(PROCESSED_DIR / "loudness.npy", loudness)

print(f"Processed files saved to: {PROCESSED_DIR}")


# Train your DDSP model

In [ ]:
STEPS = 1000 #For testing, for training use at least 50k
BATCH = 8
LR = 1e-3

### Creating the dataset
We now load the preprocessed arrays from `PROCESSED_DIR`. If you already ran the preprocessing once and the files are still there, you can restart directly from this section.


In [ ]:
class Dataset(torch.utils.data.Dataset):

    def __init__(self, out_dir):
        super().__init__()
        self.signals = np.load(os.path.join(out_dir, "signals.npy"))
        self.pitchs = np.load(os.path.join(out_dir, "pitchs.npy"))
        self.loudness = np.load(os.path.join(out_dir, "loudness.npy"))

        print(self.signals.shape)

    def __len__(self):
        return self.signals.shape[0]

    def __getitem__(self, idx):
        s = torch.from_numpy(self.signals[idx])
        p = torch.from_numpy(self.pitchs[idx])
        l = torch.from_numpy(self.loudness[idx])
        return s, p, l


def mean_std_loudness(dataset):
    mean = 0
    std = 0
    n = 0
    for _, _, l in dataset:
        n += 1
        mean += (l.mean().item() - mean) / n
        std += (l.std().item() - std) / n
    return mean, std


dataset = Dataset(PROCESSED_DIR)

dataloader = torch.utils.data.DataLoader(
    dataset,
    BATCH,
    True,
    drop_last=True,
)

mean_loudness, std_loudness = mean_std_loudness(dataloader)


Let's check the dataset

In [ ]:
s, p, l = dataset[0]

display(Audio(s.numpy().flatten(), rate=SR))

f, ax = plt.subplots(1, 2, figsize=(15, 3))
ax[0].plot(p.numpy().flatten())
ax[0].set_ylabel("Pitch")

ax[1].plot(l.numpy().flatten())
ax[1].set_ylabel("Loudness")

## Creating the model

In [ ]:
from ddsp.model import DDSP
from ddsp.core import multiscale_fft, safe_log


# Create a directory to save the model weights
run_dir = RUNS_ROOT / MODEL_NAME
run_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
model = DDSP(hidden_size=CAPACITY, sampling_rate=SR,
             block_size=BLOCK_SIZE)

If a previous training already produced a checkpoint in `run_dir`, you can reload it here and continue from the saved weights.


In [ ]:


# pretrained = torch.load(run_dir / "state.pth",
#                         map_location="cpu")
# model.load_state_dict(pretrained)
# print(None)

# model = model.to(device)


## Training Loop

In [ ]:
# Send the model to gpu
model = model.to(device)

# Initialize the optimizer
opt = torch.optim.Adam(model.parameters(), lr=LR)

# Loop variables
step = 0
mean_loss = 0
progress_bar = tqdm(range(STEPS))

# Training loop
while step < STEPS:
    for s, p, l in dataloader:
        s = s.to(device)
        p = p.unsqueeze(-1).to(device)
        l = l.unsqueeze(-1).to(device)

        l = (l - mean_loudness) / std_loudness

        y = model(p, l).squeeze(-1)

        ## Compute the Loss over stft of input and ouputs
        ori_stft = multiscale_fft(
            s,
            [2048, 1024, 512, 256, 128],
            0.75,
        )
        rec_stft = multiscale_fft(
            y,
            [2048, 1024, 512, 256, 128],
            0.75,
        )

        loss = 0
        for s_x, s_y in zip(ori_stft, rec_stft):
            lin_loss = (s_x - s_y).abs().mean()
            log_loss = (safe_log(s_x) - safe_log(s_y)).abs().mean()
            loss = loss + lin_loss + log_loss

        # Apply gradient descent to the weights of the model
        opt.zero_grad()
        loss.backward()
        opt.step()

        # Update the progress bar
        step += 1
        progress_bar.update(1)

        mean_loss += (loss.item() - mean_loss) / 100

        # Display the loss every 10 steps
        if (step + 1) % 10 == 0:
            progress_bar.set_description(f"Loss: {mean_loss:.2f}")
            mean_loss = 0

        # Save the model every 200 steps
        if (step + 1) % 500 == 0:
            torch.save(
                model.state_dict(),
                os.path.join(run_dir, "state.pth"),
            )

            audio = torch.cat([s[:2], y[:2]],
                              -1).reshape(-1).detach().cpu().numpy()
            display(Audio(audio, rate=SR))

Now that the model is trained, test the inference on one sample from the dataset. This is a quick sanity check before moving to export.


In [ ]:
s, p, l = dataset[10]

p = p.reshape(1,-1,1).to(device)
l = l.reshape(1,-1,1).to(device)
l = (l - mean_loudness) / std_loudness

y = model(p.cuda(), l.cuda()).squeeze().cpu().detach().numpy()

print("Original signal")
display(Audio(s.numpy(), rate = SR))

print("Reconstruction")
display(Audio(y, rate = SR))


f, ax = plt.subplots(1, 2, figsize=(15, 3))
ax[0].plot(p.cpu().numpy().flatten())
ax[0].set_ylabel("Pitch")

ax[1].plot(l.cpu().numpy().flatten())
ax[1].set_ylabel("Loudness")

# Exports
Export the trained DDSP model and the CREPE-related components to `torchscript` files for real-time synthesis in MaxMSP with `nn_tilde`.


### Load the trained DDSP checkpoint
We reload the saved weights from `run_dir` before building the real-time wrapper.


In [ ]:
model = DDSP(hidden_size=CAPACITY, sampling_rate=SR, block_size=BLOCK_SIZE)

run_dir = RUNS_ROOT / MODEL_NAME

pretrained = torch.load(run_dir / "state.pth",
                        map_location="cpu")
model.load_state_dict(pretrained)
model.eval()
print(None)


### Real-time DDSP Class

Now we define a wrapper around the DDSP model for real-time inference. The scripted class specifies the `forward` method that will be called by `nn_tilde` in MaxMSP.


In [ ]:
class ScriptDDSP(nn_tilde.Module):

    def __init__(self, ddsp_model, mean_loudness: float, std_loudness: float):
        super().__init__()

        self.ddsp = ddsp_model
        self.ddsp.gru.flatten_parameters()

        self.register_buffer("mean_loudness", torch.tensor(mean_loudness))
        self.register_buffer("std_loudness", torch.tensor(std_loudness))

        self.register_method(
            "forward",
            in_channels=2,
            in_ratio=self.ddsp.block_size,
            out_channels=1,
            out_ratio=1,
            input_labels=[
                f"(signal) Pitch",
                f"(signal) Loudness",
            ],
            output_labels=[f"(signal) audio out"],
            test_buffer_size=self.ddsp.block_size,
        )

    @torch.jit.export
    def forward(self, x):
        n = x.shape[0]

        pitch, loudness = torch.split(x, 1, dim=1)
        pitch = pitch.squeeze(1)
        loudness = loudness.squeeze(1)

        pitch = pitch.reshape(1, -1, 1)
        loudness = loudness.reshape(1, -1, 1)
        loudness = (loudness - self.mean_loudness) / self.std_loudness

        out = self.ddsp.realtime_forward(pitch=pitch, loudness=loudness)

        # Consider batch
        out = out.reshape(n, 1, -1)
        return out


In [ ]:
scripted_model = ScriptDDSP(
    ddsp_model=model,
    mean_loudness=mean_loudness,
    std_loudness=std_loudness,
)

Let's test and compare the scripted model inference with the normal model.

In [ ]:
s, p, l = dataset[39]

p = p.reshape(1,-1,1)
l = l.reshape(1,-1,1)
l = (l - mean_loudness) / std_loudness

y = model(p, l).squeeze().cpu().detach().numpy()

print("Original signal")
display(Audio(s.numpy(), rate = SR))

print("Reconstruction")
display(Audio(y, rate = SR))


f, ax = plt.subplots(1, 2, figsize=(15, 3))
ax[0].plot(p.cpu().numpy().flatten())
ax[0].set_ylabel("Pitch")

ax[1].plot(l.cpu().numpy().flatten())
ax[1].set_ylabel("Loudness")

s, p, l = dataset[10]

p = p.reshape(1,-1,1)
l = l.reshape(1,-1,1)


p, l = p.permute(0,2,1), l.permute(0,2,1)



outputs=[]
T = p.shape[-1]
for t in range(T):
    pitch_chunk = p[:, :, t:t+1]
    loud_chunk  = l[:, :, t:t+1]

    x = torch.cat((pitch_chunk, loud_chunk), 1)

    y = scripted_model.forward(x)

    outputs.append(y)

y = torch.cat(outputs, -1)



y = model.reverb(y.permute(0,2,1))
print("Reconstruction")
display(Audio(y.detach().squeeze(), rate = SR))


Now we can save the scripted model to a `.ts` file for MaxMSP inference. The export will be written inside `EXPORTS_DIR`.


In [ ]:
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
path = os.path.join(EXPORTS_DIR, MODEL_NAME+".ts")
scripted_model.export_to_ts(path)
print(f"Model exported to: {path}")

Almost done: we also need the impulse response parameters. They are extracted from the trained reverb and saved next to the TorchScript export.


In [ ]:
impulse = scripted_model.ddsp.reverb.build_impulse().reshape(
    -1).detach().numpy()

path = os.path.join(EXPORTS_DIR, "impulse_response_" + MODEL_NAME + ".wav")

sf.write(
    path,
    impulse,
    SR,
)

print(f"Impulse response exported to: {path}")